# Exploration

Quick look at the warehouse built by `make data`. Numbers here are for exploration only; reported numbers come from the pipeline tables.

In [ ]:
import sys; sys.path.insert(0, '../src')
import duckdb, matplotlib.pyplot as plt
from common import load_config, ROOT
con = duckdb.connect(str(ROOT / load_config()['paths']['warehouse']), read_only=True)

In [ ]:
con.sql("SELECT table_name FROM information_schema.tables ORDER BY 1").df()

## Total daily units by category

In [ ]:
daily = con.sql("""SELECT s.date, i.cat_id, sum(s.units) AS units
  FROM fact_sales s JOIN dim_item i USING (item_id) GROUP BY ALL ORDER BY s.date""").df()
daily.pivot(index='date', columns='cat_id', values='units').rolling(7).mean().plot(figsize=(11, 4), title='7-day mean units by category');

## Zero-sales share per series (post-launch)

In [ ]:
z = con.sql("SELECT item_id, store_id, avg((units = 0)::INT) AS zero_share FROM v_sales_clean GROUP BY ALL").df()
z.zero_share.plot(kind='hist', bins=50, figsize=(8, 3.5), title='Share of zero-sales days per series');

## Example series from the sample

In [ ]:
ex = con.sql("""SELECT f.date, f.units FROM fact_sales f JOIN sample_series s USING (item_id, store_id)
  WHERE s.series_idx = 0 AND f.d >= 1578 ORDER BY f.date""").df()
ex.plot(x='date', y='units', figsize=(11, 3), title='Sample series 0, simulation window');